# 01 — Cognitive Substrate

**Goal:** build an agent that knows when it is unsure of itself.

In this notebook you will:

1. Make an agent that rates its own answer (a confidence score from 0 to 1).
2. Send that score to an event bus.
3. Watch the scores over time. If they drop, stop the run.

That is the full idea: the agent reports how sure it is, and a watcher decides what to do about it.

**Before you start:** you need a `.env` file at the repo root with `LLM_API_KEY` and `LLM_MODEL`. See the main README.


## 1. Load the config

`load_config()` reads your `.env` and returns an `OrqestConfig` object. We only need the model name and the API key.


In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. Define what the agent returns

Most agents return just an answer. We want a bit more.

We will ask the agent to return four fields:

- `answer` — the actual answer.
- `self_confidence` — a number from 0 to 1. How sure is the agent?
- `uncertain_about` — a list of things the agent is unsure about.
- `outside_my_capability` — `True` if the question cannot be answered (for example, "what will happen next week?").

The agent fills in all four fields in one LLM call. No extra cost.


In [2]:
from pydantic import BaseModel, Field


class AnswerOutput(BaseModel):
    """What the agent returns: an answer plus a self-rating."""

    answer: str = Field(description="The answer to the user's question.")

    self_confidence: float | None = Field(
        default=None,
        description="Your honest probability (0-1) that this answer is correct.",
    )

    uncertain_about: list[str] = Field(
        default_factory=list,
        description="Things you are unsure about.",
    )

    outside_my_capability: bool = Field(
        default=False,
        description=(
            "True if the question needs information you cannot have "
            "(future events, private data, post-cutoff knowledge)."
        ),
    )

## 3. Build the agent

We subclass `BaseAgent`. Two things to notice:

- `output_type=AnswerOutput` — the agent must return our four-field model.
- `confidence_protocol=StructuredOutputProtocol()` — this tells orqest where the confidence score lives. The protocol reads it straight off the output. Zero extra LLM calls.

The system prompt asks the agent to be honest about its uncertainty. This matters: without it, the model tends to mark everything as 0.95.


In [3]:
from orqest.agents import BaseAgent, GlobalState
from orqest.metacognition import StructuredOutputProtocol


class ResearchAgent(BaseAgent[GlobalState, AnswerOutput]):
    def __init__(self, model, api_key: str | None = None):
        super().__init__(
            agent_name="research_agent",
            system_prompt=(
                "You answer questions concisely. Be honest in your self-assessment: "
                "set self_confidence LOW when you are guessing at a specific fact, "
                "list what you are unsure about, and set outside_my_capability=True "
                "when the question needs information no one could have yet."
            ),
            output_type=AnswerOutput,
            model=model,
            api_key=api_key,
            confidence_protocol=StructuredOutputProtocol(),
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> AnswerOutput:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


agent = ResearchAgent(model=config.llm_model, api_key=config.llm_api_key)
print("Agent built.")

Agent built.


## 4. Ask the agent something easy

`run` returns just the `AnswerOutput`. `run_enriched` returns the same output **plus** a normalised confidence score, the list of uncertainties, and a `capability_boundary` flag.

Think of `run_enriched` as "run, and also report how you feel about the answer". Let's try a simple question first.


In [6]:
async def ask(question: str):
    """Ask the agent a question and print the enriched result."""
    state = GlobalState()
    state.add_message("user", question)
    enriched = await agent.run_enriched(state)
    print(f"Q: {question}")
    print(f"   answer:     {enriched.output.answer[:250]}")
    print(f"   confidence: {enriched.confidence}")
    print(f"   uncertain:  {enriched.uncertainty_targets}")
    print(f"   boundary:   {enriched.capability_boundary}")
    return enriched

In [7]:
_ = await ask("What is the capital of France?")

Q: What is the capital of France?
   answer:     Paris.
   confidence: 0.99
   uncertain:  []
   boundary:   False


Now let's try a question that **cannot** be answered. The agent should say so.

In [8]:
_ = await ask("What will the GBP/USD exchange rate be next Tuesday?")

Q: What will the GBP/USD exchange rate be next Tuesday?
   answer:     I can’t reliably predict the GBP/USD exchange rate for next Tuesday. Exchange rates are driven by real‑time market news, macro data releases, and sentiment, so any single-number forecast would be speculative.

If you tell me which Tuesday (date and y
   confidence: 0.95
   uncertain:  []
   boundary:   True


### What just happened?

Two things to notice — and the second one is important:

- For the impossible question, `capability_boundary` is **True**. This is a clear, reliable signal: "I cannot answer this."
- `confidence` may still be **high** on that same question. Why? Because the agent is confident that *"I can't know this"* is the right response.

**Key lesson:** `confidence` measures how sure the agent is about its response, not how hard the question is. When you build a healing system, the `capability_boundary` flag is often more useful than the confidence number on its own.


## 5. Send confidence to an event bus

So far the confidence score just gets printed. Now we want to **watch** it over many runs.

We use three pieces:

- `EventBus` — a simple publish/subscribe channel.
- `MetacognitionHook` — when a tool returns an `EnrichedOutput`, it publishes a `metacognition.confidence` event.
- `RegressionDetector` — listens for those events. Buffers a sliding window of scores. Fires if the scores drop.

In a real run, the hook is called automatically inside a `CompoundTool`. Here we call it by hand so you can see the wiring.


In [9]:
from orqest.observability import EventBus
from orqest.metacognition import MetacognitionHook
from orqest.healing import RegressionDetector

bus = EventBus()

# The detector subscribes to confidence events on this bus.
detector = RegressionDetector(window_n=4, drop_threshold=0.15)
detector.subscribe(bus)

# The hook publishes confidence events to the same bus.
metacog_hook = MetacognitionHook(bus, agent_name="research_agent")

print("Bus, detector, and hook are connected.")

Bus, detector, and hook are connected.


Now run a few prompts and feed each result into the hook by hand. This is exactly what orqest does for you inside a compound tool flow.

In [10]:
prompts = [
    "What is the boiling point of water at sea level, in Celsius?",
    "Summarise the main causes of the 2008 financial crisis in two sentences.",
    "Name the single most valuable startup that will be founded next year.",
]

for p in prompts:
    state = GlobalState()
    state.add_message("user", p)
    enriched = await agent.run_enriched(state)

    # Hand the result to the hook — same as a real compound tool flow.
    await metacog_hook.after_tool("research", {}, enriched, state, 0.0)

    print(
        f"confidence={enriched.confidence}  boundary={enriched.capability_boundary}"
        f"  ::  {p[:150]}"
    )

confidence=0.99  boundary=False  ::  What is the boiling point of water at sea level, in Celsius?
confidence=0.9  boundary=False  ::  Summarise the main causes of the 2008 financial crisis in two sentences.
confidence=0.95  boundary=True  ::  Name the single most valuable startup that will be founded next year.


## 6. Detect a drop and stop the run

Real LLM confidence is noisy. To see the detector fire reliably, we feed it a controlled sequence of scores that drops: `0.97, 0.92, 0.45, 0.20`. In production these scores arrive live from the hook.

We then ask the hook system: "should the next step run, or should we abort?" The detector says **abort** because the confidence trend is bad.


In [11]:
from orqest.observability import AgentEvent

# Fresh bus and detector so the demo is isolated.
demo_bus = EventBus()
demo_detector = RegressionDetector(window_n=4, drop_threshold=0.15)
demo_detector.subscribe(demo_bus)

# Emit a declining-confidence sequence onto the bus.
for conf in [0.97, 0.92, 0.45, 0.20]:
    await demo_bus.emit(AgentEvent(
        event_type="metacognition.confidence",
        agent_name="research_agent",
        data={"confidence": conf},
    ))

print("Four confidence events sent. The detector is now suspicious.")

Four confidence events sent. The detector is now suspicious.


Now run the watchdog. The `WatchdogHook` asks each detector for a signal. If a detector reports a problem, the default policy returns an `AbortRun` decision. In a `CompoundTool` flow this raises `HookAbortError`, which stops the run.

In [12]:
from orqest.healing import WatchdogHook, default_policy
from orqest.hooks import HookRunner, HookAbortError

runner = HookRunner([WatchdogHook([demo_detector], policy=default_policy)])

try:
    await runner.run_before("next_research_step", {}, None)
    print("no halt — confidence trend is stable")
except HookAbortError as exc:
    print(f"HookAbortError raised — {exc}")

HookAbortError raised — regression: Confidence dropped from 0.95 → 0.33


## 7. Bonus — model fallback

The `RegressionDetector` watches **the agent**. But what if **the provider** has a problem? A timeout, a 500 error, a rate limit?

`FallbackModel` wraps a list of models. If the first one fails with a transient error, it switches to the next one. It is a normal `pydantic_ai.Model`, so it drops straight into any agent.

`resolve_model_with_fallback` builds one from a list of `provider:model_id` strings. We pass a list of one here just to show the wiring — in real code you would pass two or more.


In [13]:
from orqest.healing import resolve_model_with_fallback

fallback_model = resolve_model_with_fallback(
    [config.llm_model],
    api_key=config.llm_api_key,
)
print(f"fallback model: {fallback_model.model_name}")

# The fallback model slots straight into the agent. The agent loop is unchanged.
resilient_agent = ResearchAgent(model=fallback_model)
print(f"agent model:    {resilient_agent.model.model_name}")

fallback model: fallback(gpt-5.2)
agent model:    fallback(gpt-5.2)


## Recap

Here is the whole picture in one short list:

1. The agent returns an answer **and** a self-rating in the same call. No extra cost.
2. `run_enriched` gives you that self-rating in a normalised shape (`EnrichedOutput`).
3. A `MetacognitionHook` sends the confidence score to an `EventBus`.
4. A `RegressionDetector` watches the scores. If they drop, it fires.
5. A `WatchdogHook` turns a fired detector into a "stop the run" decision.
6. `FallbackModel` handles the other axis: provider failures.

**Two honest notes:**

- Raw LLM confidence is noisy. Treat single scores with caution. Trends are more reliable than absolutes.
- `capability_boundary` is often a better signal than the raw confidence number, because it answers a clearer question: "is this answerable at all?"

In production you do not call the watchdog by hand. A `HealingRunner` (or `Workbench.with_healing(config)`) owns the bus subscriptions and a poll loop. This notebook unrolled that loop so each step was visible.
